# Step 5 — group-level cross-cohort brain <-> behavior

The SONA behavioral cohort (Track A: run-resolved `positive emotion`, `like`, PC1) are different people
from the fMRI cohort, so they reach the brain only at the **group level**, via the 3 scramble groups:
relate a group-averaged neural quantity at `(group, character, run)` to the group-level behavioral target
at the same grid, for **both** PCA (PC1) and non-PCA (positive emotion / like) from `01__targets.csv`.

**Open decision for Hayoung — the neural quantity.** Default = group-mean inter-subject ISC per
`(group, char, run)` (characters reordered via `rearrange_new`, same as `04b`). Alternatives: neural
pattern distance (Fig-1 style) or event discriminability. Confirm before interpreting.

## 5.0 · Paths, targets, ported neural helpers

In [5]:
import os, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from config import JIN_REPO
NEURAL_PATH = os.path.join(JIN_REPO, "data/brain/similarity/neuralISC_byevent.npy")
from config import EVENT_PATH
TARGETS_CSV = "results/step1/01__targets.csv"          # from 01 §1.6b (PCA + non-PCA columns)
# -------------------------
HAVE_TARGETS = True
HAVE_BRAIN   = True
NROI = 116; N_BOOT = 10000
CHARS = ["jack","kate","randall","kevin"]
BOTH_TARGETS = ["PC1", "positive_emotion", "like"]     # PC1 = PC1 of the 16-item SONA survey (Track A, group-level); NOT the 35-item post-scan PC1_PCA used in 04c
print("targets present:", HAVE_TARGETS, "| Jin neural+events present:", HAVE_BRAIN)

from helpers import *

if HAVE_TARGETS:
    targets=pd.read_csv(TARGETS_CSV); targets["Character"]=targets["Character"].str.lower().str.strip()
    print("target grid:", targets.shape)
else:
    targets=None; print("Run 01 first to create", TARGETS_CSV)

targets present: True | Jin neural+events present: True
target grid: (120, 8)


## 5.1 · Group-level neural quantity per (group, character, run)

Group-mean inter-subject ISC for each `(roi, group, run)`, characters reordered via `rearrange_new`
(same as `04b`), averaged across subject-pairs to a group scalar per `(roi, group, character, run)`.

In [6]:
def neural_group_grid(nroi=NROI):
    neural=np.load(NEURAL_PATH, allow_pickle=True).item()
    out={}
    for roi in range(1, nroi+1):
        rows=[]
        for g in [1,2,3]:
            for r in range(10):
                tb=neural[roi,g,r]
                if r==6:                                   # 7th run: no character structure -> group mean, all chars
                    val=np.nanmean(tb); per_char=[val]*4
                else:
                    per_char=np.nanmean(rearrange_new(g, r+1, tb), axis=1)   # (4,)
                for ci in range(4):
                    rows.append({"group":g,"Character":CHARS[ci],"Run":r+1,"neural":float(per_char[ci])})
        out[roi-1]=pd.DataFrame(rows)
    return out

## 5.2 · Group-level brain<->behavior per ROI, per target (real run)

Per ROI, Spearman the neural scalar vs each behavioral target across the `(group, char, run)` cells, with
a bootstrap p and FDR across ROIs. Run once per target (PCA + non-PCA carried together). Clustering
caveat: only 3 groups — treat as descriptive.

In [7]:
def group_brain_behavior(grid, target_df, target_col, nroi):
    r_all=[]; p_all=[]
    for roi in range(nroi):
        m=grid[roi].merge(target_df[["group","Character","Run",target_col]],
                          on=["group","Character","Run"]).dropna(subset=["neural",target_col])
        if len(m)<5: r_all.append(np.nan); p_all.append(1.0); continue
        r_all.append(spearmanr(m["neural"], m[target_col])[0])
        rng=np.random.default_rng(roi)
        bs=[]
        arr=m[["neural",target_col]].to_numpy()
        for _ in range(500):
            s=arr[rng.integers(0,len(arr),len(arr))]
            bs.append(spearmanr(s[:,0], s[:,1])[0])
        bs=np.array(bs); p_all.append(min(2*min(np.mean(bs<=0), np.mean(bs>=0)),1.0))
    _,pc,_,_=multipletests(np.nan_to_num(p_all,nan=1.0), alpha=0.05, method="fdr_bh")
    return np.array(r_all), np.array(p_all), pc, np.where(pc<0.05)[0]

if HAVE_BRAIN and HAVE_TARGETS:
    grid=neural_group_grid()
    os.makedirs("results/step5", exist_ok=True)
    for tgt in BOTH_TARGETS:
        r,p,pc,sig=group_brain_behavior(grid, targets, tgt, NROI)
        np.save(f"results/step5/05__group_brain_{tgt}.npy", {"r":r,"p":p,"p_fdr":pc,"sig_rois":sig}, allow_pickle=True)
        print(f"[{tgt:16s}] FDR-sig ROIs (q<0.05): {list(sig)}  | max|r|={np.nanmax(np.abs(r)):.3f}")
else:
    print("Set the Jin neural/event paths (and run 01 for targets), then re-run to produce 05 results.")

[PC1             ] FDR-sig ROIs (q<0.05): []  | max|r|=0.236
[positive_emotion] FDR-sig ROIs (q<0.05): []  | max|r|=0.240
[like            ] FDR-sig ROIs (q<0.05): [np.int64(91)]  | max|r|=0.299


### Finding (rerun) — group-level `like` also moves with the brain
- **`like` → 1 FDR-sig ROI [91]** (q<0.05), max|r| 0.299; **`PC1` → 0, `positive_emotion` → 0** (max|r|
  ~0.24). Same direction as `04c`: shared *liking* co-varies with a group neural quantity; sentiment/PC1 don't.
- Coarse (3 scramble groups, cross-cohort) — descriptive, reported beside `04c`. ROI [91] needs a name + replication.

## ASK HAYOUNG!!
- **ASK HAYOUNG!!** Neural quantity for the group-level bridge: group-mean ISC vs neural-pattern distance vs event discriminability. *(The reported run in 5.1–5.2 uses the default — group-mean ISC — which gave `like` -> ROI [91]; the choice is still open for confirmation.)*

## 5.3 · What to confirm with Hayoung before trusting 05
- **The neural quantity** (group-mean ISC vs pattern distance vs discriminability) — the defining choice.
- **Only 3 groups**: descriptive; do not over-interpret any single ROI.
- Report **beside** the subject-level `04b`/`04c`, never pooled (different cohorts and resolutions).

## 5.4 · Positive-valence aggregates at group level (added alongside)

`char_valence_composite` and `affect_cluster` were built in Step 1 but never run through the brain. **Result:** `char_valence_composite` → **[101] RH Amygdala** (q<0.05, max|r| 0.229); `affect_cluster` → none. So the group-level positive-valence aggregate lands in **amygdala (affect)**, distinct from `like`'s **[91] RH DMN medial PFC** — a valence-vs-viewer-stance dissociation even within the positive family.

In [8]:
# 5.4 · Positive-valence AGGREGATES at group level (char_valence_composite, affect_cluster).
# These composites were built in Step 1 but never carried through to 05 -- run them ALONGSIDE PC1/positive_emotion/like.
import numpy as np, pandas as pd, os
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from config import NEURAL_PATH
from helpers import rearrange_new
NROI=116; CHARS=["jack","kate","randall","kevin"]
targets=pd.read_csv("results/step1/01__targets.csv"); targets["Character"]=targets["Character"].str.lower().str.strip()
def _grid():
    neural=np.load(NEURAL_PATH,allow_pickle=True).item(); out={}
    for roi in range(1,NROI+1):
        rows=[]
        for g in [1,2,3]:
            for r in range(10):
                tb=neural[roi,g,r]; per=[np.nanmean(tb)]*4 if r==6 else np.nanmean(rearrange_new(g,r+1,tb),axis=1)
                for ci in range(4): rows.append({"group":g,"Character":CHARS[ci],"Run":r+1,"neural":float(per[ci])})
        out[roi-1]=pd.DataFrame(rows)
    return out
def _gbb(grid,col):
    r_all=[];p_all=[]
    for roi in range(NROI):
        m=grid[roi].merge(targets[["group","Character","Run",col]],on=["group","Character","Run"]).dropna(subset=["neural",col])
        if len(m)<5: r_all.append(np.nan);p_all.append(1.0);continue
        r_all.append(spearmanr(m["neural"],m[col])[0]); rng=np.random.default_rng(roi); arr=m[["neural",col]].to_numpy()
        bs=np.array([spearmanr(*arr[rng.integers(0,len(arr),len(arr))].T)[0] for _ in range(500)])
        p_all.append(min(2*min(np.mean(bs<=0),np.mean(bs>=0)),1.0))
    _,pc,_,_=multipletests(np.nan_to_num(p_all,nan=1.0),alpha=0.05,method="fdr_bh")
    return np.array(r_all),pc,np.where(pc<0.05)[0]
grid=_grid()
for tgt in ["char_valence_composite","affect_cluster"]:
    r,pc,sig=_gbb(grid,tgt)
    np.save(f"results/step5/05__group_brain_{tgt}.npy",{"r":r,"p_fdr":pc,"sig_rois":sig},allow_pickle=True)
    print(f"[{tgt:22s}] FDR-sig (q<0.05): {[int(x) for x in sig]}  | max|r|={np.nanmax(np.abs(r)):.3f}")
print("reference: like [91], PC1 [], positive_emotion []")

[char_valence_composite] FDR-sig (q<0.05): [101]  | max|r|=0.229
[affect_cluster        ] FDR-sig (q<0.05): []  | max|r|=0.229
reference: like [91], PC1 [], positive_emotion []
